# NusaSR: fine-tune Real-ESRGAN x4v3 untuk gambar LoRa

Melatih lanjut **Real-ESRGAN General x4v3** dengan kerusakan yang sama seperti gambar NusaMesh: diperkecil ke 160/256/384 px lalu WebP 2/4/8 KB.

**Cara pakai:** Runtime → Change runtime type → **GPU**, lalu Run all. Checkpoint disimpan di Google Drive folder `NusaSR`, jadi kalau Colab putus cukup Run all lagi dan training lanjut.

**Foto lapangan (opsional, sangat disarankan):** taruh di Drive `NusaSR/foto_lapangan`. Foto ini hanya dipakai untuk uji, tidak ikut dilatih.

**Hasil di Drive:** `best.pt` dan `real_esrgan_x4v3_lora_w8a8.tflite`, plus `hasil.txt` berisi perbandingan dengan bicubic dan model bawaan. Pakai loss L1 tanpa GAN supaya model tidak mengarang detail.

## 1. Persiapan

In [ ]:
# COLAB_ONLY
!pip install -q lpips scikit-image
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# CONFIG
import glob, io, math, os, random, time, urllib.request, zipfile
from multiprocessing.pool import ThreadPool
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset

WORK = Path("/content/nusasr")
SAVE = Path("/content/drive/MyDrive/NusaSR")
TRAIN_HR = WORK / "DIV2K_train_HR"
VAL_HR = WORK / "DIV2K_valid_HR"
FIELD_HR = SAVE / "foto_lapangan"
WEIGHTS = WORK / "realesr-general-x4v3.pth"

PRESETS = [(160, 2000), (256, 4000), (384, 8000)]  # (sisi terpanjang px, batas byte), sama dengan LoraImageQuality
SCALE = 4
LR_PATCH = 64
ITERS = 20000
BATCH = 16
LEARNING_RATE = 1e-4
NUM_WORKERS = 2
VAL_EVERY = 2000
VAL_PAIRS = 60          # jumlah pasangan untuk validasi saat training
TFLITE_IMAGES = 10      # jumlah gambar untuk uji model TFLite 8-bit
CALIBRATION = 200       # jumlah potongan untuk kuantisasi
OVERLAP = 16            # sama dengan SrTiling.OVERLAP di app
THREADS = 4
SEED = 42

In [ ]:
for d in (WORK, SAVE):
    d.mkdir(parents=True, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "(tanpa GPU: ganti runtime ke GPU)")

In [ ]:
# COLAB_ONLY
def fetch(url, dest):
    if not Path(dest).exists():
        print("unduh", url)
        urllib.request.urlretrieve(url, dest)

BASE = "http://data.vision.ee.ethz.ch/cvl/DIV2K"
for name in ("DIV2K_train_HR", "DIV2K_valid_HR"):
    if not (WORK / name).exists():
        z = WORK / f"{name}.zip"
        fetch(f"{BASE}/{name}.zip", z)
        with zipfile.ZipFile(z) as f:
            f.extractall(WORK)
        z.unlink()
fetch("https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.5.0/realesr-general-x4v3.pth", WEIGHTS)

## 2. Model

In [ ]:
class SRVGGNetCompact(nn.Module):
    """Arsitektur Real-ESRGAN General x4v3 (32 konvolusi, 64 fitur, PReLU)."""
    def __init__(self, feats=64, convs=32, scale=SCALE):
        super().__init__()
        self.scale = scale
        body = [nn.Conv2d(3, feats, 3, 1, 1), nn.PReLU(feats)]
        for _ in range(convs):
            body += [nn.Conv2d(feats, feats, 3, 1, 1), nn.PReLU(feats)]
        body.append(nn.Conv2d(feats, 3 * scale * scale, 3, 1, 1))
        self.body = nn.ModuleList(body)
        self.upsampler = nn.PixelShuffle(scale)

    def forward(self, x):
        out = x
        for layer in self.body:
            out = layer(out)
        return self.upsampler(out) + F.interpolate(x, scale_factor=self.scale, mode="nearest")


def load_pretrained():
    model = SRVGGNetCompact()
    model.load_state_dict(torch.load(WEIGHTS, map_location="cpu", weights_only=True)["params"])
    return model

## 3. Kerusakan seperti di app
Mengikuti `LoraImageCodec.kt`: `inSampleSize` pangkat 2, resize bilinear, lalu cari kualitas WebP tertinggi yang muat batas byte. Bila kualitas 5 pun tidak muat, sisi gambar diperkecil 15%.

In [ ]:
QUALITY_MIN, QUALITY_MAX, MIN_SIDE, SIDE_SHRINK = 5, 90, 64, 0.85


def android_downscale(src, target):
    long_side = max(src.size)
    sample = 1
    while long_side // (sample * 2) >= target * 2:
        sample *= 2
    img = src.reduce(sample) if sample > 1 else src
    s = target / max(img.size)
    return img.resize((max(1, int(img.width * s + 0.5)), max(1, int(img.height * s + 0.5))), Image.BILINEAR)


def webp_bytes(img, quality):
    buf = io.BytesIO()
    img.save(buf, "WEBP", quality=quality)
    return buf.getvalue()


def lora_degrade(src, max_side, max_bytes):
    side = max_side
    while True:
        small = android_downscale(src, side)
        lo, hi, best = QUALITY_MIN, QUALITY_MAX, None
        while lo <= hi:
            mid = (lo + hi) // 2
            data = webp_bytes(small, mid)
            if len(data) <= max_bytes:
                best, lo = data, mid + 1
            else:
                hi = mid - 1
        if best is not None or side <= MIN_SIDE:
            data = best if best is not None else webp_bytes(small, QUALITY_MIN)
            return Image.open(io.BytesIO(data)).convert("RGB")
        side = max(MIN_SIDE, int(side * SIDE_SHRINK))


def make_pair(args):
    path, out, max_side, max_bytes = args
    stem = Path(path).stem.replace("_", "-")
    lr_path, gt_path = out / f"{stem}_{max_side}_lr.npy", out / f"{stem}_{max_side}_gt.npy"
    if lr_path.exists() and gt_path.exists():
        return str(lr_path)
    src = Image.open(path).convert("RGB")
    lr = lora_degrade(src, max_side, max_bytes)
    size = (lr.width * SCALE, lr.height * SCALE)
    if size[0] > src.width or size[1] > src.height or min(lr.size) < LR_PATCH:
        return None  # foto terlalu kecil untuk jadi kunci jawaban tanpa diperbesar
    np.save(gt_path, np.asarray(src.resize(size, Image.LANCZOS), dtype=np.uint8))
    np.save(lr_path, np.asarray(lr, dtype=np.uint8))
    return str(lr_path)


def build_pairs(hr_dir, out):
    out.mkdir(parents=True, exist_ok=True)
    files = []
    for ext in ("png", "jpg", "jpeg", "webp", "PNG", "JPG", "JPEG"):
        files += glob.glob(f"{hr_dir}/**/*.{ext}", recursive=True)
    files = sorted(set(files))
    jobs = [(p, out, side, size) for p in files for side, size in PRESETS]
    with ThreadPool(os.cpu_count() or 2) as pool:
        made = [p for p in pool.imap_unordered(make_pair, jobs) if p]
    print(f"{hr_dir}: {len(files)} foto, {len(made)} pasangan")
    return sorted(made)


train_pairs = build_pairs(TRAIN_HR, WORK / "pairs_train")
val_pairs = build_pairs(VAL_HR, WORK / "pairs_val")
field_pairs = build_pairs(FIELD_HR, WORK / "pairs_field") if FIELD_HR.exists() else []
assert train_pairs and val_pairs, "dataset kosong"

In [ ]:
def gt_path(lr_path):
    return lr_path[:-len("lr.npy")] + "gt.npy"


def preset_of(lr_path):
    return int(Path(lr_path).stem.rsplit("_", 2)[1])


def to_tensor(a):
    return torch.from_numpy(a.copy()).permute(2, 0, 1).float() / 255.0


class Patches(Dataset):
    def __init__(self, pairs, length):
        self.pairs, self.length = pairs, length

    def __len__(self):
        return self.length

    def __getitem__(self, _):
        p = random.choice(self.pairs)
        lr, gt = np.load(p, mmap_mode="r"), np.load(gt_path(p), mmap_mode="r")
        y, x = random.randint(0, lr.shape[0] - LR_PATCH), random.randint(0, lr.shape[1] - LR_PATCH)
        a = np.array(lr[y:y + LR_PATCH, x:x + LR_PATCH])
        b = np.array(gt[y * SCALE:(y + LR_PATCH) * SCALE, x * SCALE:(x + LR_PATCH) * SCALE])
        if random.random() < .5:
            a, b = a[:, ::-1], b[:, ::-1]
        if random.random() < .5:
            a, b = a[::-1], b[::-1]
        if random.random() < .5:
            a, b = a.transpose(1, 0, 2), b.transpose(1, 0, 2)
        return to_tensor(a), to_tensor(b)

## 4. Metrik

In [ ]:
try:
    import lpips
    LPIPS_NET = lpips.LPIPS(net="alex", verbose=False).to(DEVICE).eval()
except Exception as e:
    LPIPS_NET = None
    print("LPIPS dilewati:", e)
try:
    from skimage.metrics import structural_similarity
except Exception as e:
    structural_similarity = None
    print("SSIM dilewati:", e)


def luma(a):
    return 16 + (65.481 * a[..., 0] + 128.553 * a[..., 1] + 24.966 * a[..., 2]) / 255.0


def psnr(a, b):
    return 10 * math.log10(255 ** 2 / max(np.mean((luma(a.astype(float)) - luma(b.astype(float))) ** 2), 1e-10))


def ssim(a, b):
    return structural_similarity(luma(a.astype(float)), luma(b.astype(float)), data_range=255) if structural_similarity else float("nan")


@torch.no_grad()
def lpips_of(a, b):
    if LPIPS_NET is None:
        return float("nan")
    t = lambda x: (to_tensor(x)[None].to(DEVICE) * 2 - 1)
    return float(LPIPS_NET(t(a), t(b)))


def to_uint8(t):
    return (t.clamp(0, 1)[0].permute(1, 2, 0).cpu().numpy() * 255).round().astype(np.uint8)


def bicubic(lr):
    return np.asarray(Image.fromarray(lr).resize((lr.shape[1] * SCALE, lr.shape[0] * SCALE), Image.BICUBIC))


def model_fn(model):
    @torch.no_grad()
    def run(lr):
        model.eval()
        return to_uint8(model(to_tensor(lr)[None].to(DEVICE)))
    return run


def evaluate(fn, pairs, full=True):
    rows = {}
    for p in pairs:
        lr, gt = np.load(p), np.load(gt_path(p))
        out = fn(lr)
        row = [psnr(gt, out)] + ([ssim(gt, out), lpips_of(gt, out)] if full else [])
        rows.setdefault(preset_of(p), []).append(row)
    return {k: np.mean(v, axis=0) for k, v in sorted(rows.items())}


val_sample = random.Random(SEED).sample(val_pairs, min(VAL_PAIRS, len(val_pairs)))

## 5. Training
Checkpoint `last.pt` ditulis setiap validasi. Run all lagi akan melanjutkan dari situ.

In [ ]:
model = load_pretrained().to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, betas=(.9, .99))
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=ITERS, eta_min=1e-6)
start, best = 0, -1.0
last_ckpt, best_ckpt = SAVE / "last.pt", SAVE / "best.pt"

if last_ckpt.exists():
    state = torch.load(last_ckpt, map_location=DEVICE, weights_only=True)
    model.load_state_dict(state["model"]); opt.load_state_dict(state["opt"]); sched.load_state_dict(state["sched"])
    start, best = state["iter"], state["best"]
    print(f"lanjut dari iterasi {start}, PSNR terbaik {best:.3f}")


def overall(result):
    return float(np.mean([v[0] for v in result.values()]))


if start == 0:
    best = overall(evaluate(model_fn(model), val_sample, full=False))
    print(f"awal (bobot bawaan): PSNR {best:.3f}")
    torch.save(model.state_dict(), best_ckpt)

loader = DataLoader(Patches(train_pairs, max(ITERS - start, 0) * BATCH), batch_size=BATCH, num_workers=NUM_WORKERS,
                    pin_memory=DEVICE == "cuda", persistent_workers=NUM_WORKERS > 0, drop_last=True)
model.train()
it, seen, loss_sum, t0 = start, 0, 0.0, time.time()
for lr_batch, gt_batch in loader:
    lr_batch, gt_batch = lr_batch.to(DEVICE, non_blocking=True), gt_batch.to(DEVICE, non_blocking=True)
    loss = F.l1_loss(model(lr_batch), gt_batch)
    opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); sched.step()
    it += 1; seen += 1; loss_sum += float(loss)
    if it % 200 == 0 or it == ITERS:
        eta = (time.time() - t0) / seen * (ITERS - it) / 60
        print(f"iter {it}/{ITERS}  L1 {loss_sum / min(seen, 200):.5f}  sisa ±{eta:.0f} menit", flush=True)
        loss_sum = 0.0
    if it % VAL_EVERY == 0 or it == ITERS:
        score = overall(evaluate(model_fn(model), val_sample, full=False))
        if score > best:
            best = score
            torch.save(model.state_dict(), best_ckpt)
        torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(), "iter": it, "best": best}, last_ckpt)
        print(f"validasi {it}: PSNR {score:.3f} (terbaik {best:.3f})", flush=True)
        model.train()
print("selesai, PSNR validasi terbaik:", round(best, 3))

## 6. Perbandingan
PSNR/SSIM makin tinggi makin baik, LPIPS makin rendah makin baik. Model hasil training hanya layak dipakai kalau menang di foto lapangan.

In [ ]:
tuned = SRVGGNetCompact().to(DEVICE)
tuned.load_state_dict(torch.load(best_ckpt, map_location=DEVICE, weights_only=True))
base = load_pretrained().to(DEVICE)
lines = []


def report(title, pairs, methods):
    start_line = len(lines)
    lines.append(f"\n== {title} ({len(pairs)} pasangan) ==")
    lines.append(f"{'metode':<26}{'preset':>7}{'PSNR':>9}{'SSIM':>9}{'LPIPS':>9}")
    for name, fn in methods:
        for preset, (p, s, l) in evaluate(fn, pairs).items():
            lines.append(f"{name:<26}{preset:>7}{p:>9.3f}{s:>9.4f}{l:>9.4f}")
    print("\n".join(lines[start_line:]))


methods = [("bicubic", bicubic), ("x4v3 bawaan", model_fn(base)), ("x4v3 fine-tune", model_fn(tuned))]
report("DIV2K valid", val_pairs, methods)
if field_pairs:
    report("Foto lapangan", field_pairs, methods)
else:
    lines.append("\nFoto lapangan belum ada: hasil belum mewakili kondisi nyata.")

## 7. Ekspor ke TFLite untuk app
Format sama dengan model bawaan: input `uint8` 128×128 RGB, output `uint8` 512×512 RGB.

In [ ]:
import tensorflow as tf
keras = tf.keras


class DepthToSpace(keras.layers.Layer):
    def call(self, x):
        return tf.nn.depth_to_space(x, SCALE)


def build_keras(size=128, feats=64, convs=32):
    inp = keras.Input((size, size, 3), batch_size=1)
    x, conv_layers, prelu_layers = inp, [], []
    for i in range(convs + 1):
        conv, prelu = keras.layers.Conv2D(feats, 3, padding="same"), keras.layers.PReLU(shared_axes=[1, 2])
        x = prelu(conv(x)); conv_layers.append(conv); prelu_layers.append(prelu)
    last = keras.layers.Conv2D(3 * SCALE * SCALE, 3, padding="same")
    x = DepthToSpace()(last(x))
    out = keras.layers.Add()([x, keras.layers.UpSampling2D(SCALE, interpolation="nearest")(inp)])
    return keras.Model(inp, out), conv_layers + [last], prelu_layers


def port_weights(state, conv_layers, prelu_layers):
    n = lambda k: state[k].detach().cpu().numpy()
    for i, conv in enumerate(conv_layers):
        idx = 2 * i
        kernel, bias = n(f"body.{idx}.weight").transpose(2, 3, 1, 0), n(f"body.{idx}.bias")
        if conv is conv_layers[-1]:  # PyTorch PixelShuffle (C,i,j) ke TF depth_to_space (i,j,C)
            order = [c * SCALE ** 2 + i2 * SCALE + j for i2 in range(SCALE) for j in range(SCALE) for c in range(3)]
            kernel, bias = kernel[..., order], bias[order]
        conv.set_weights([kernel, bias])
    for i, prelu in enumerate(prelu_layers):
        prelu.set_weights([n(f"body.{2 * i + 1}.weight").reshape(1, 1, -1)])


def pad_tile(a):
    return np.pad(a, ((0, 128 - a.shape[0]), (0, 128 - a.shape[1]), (0, 0)), mode="edge")


def calibration_patches():
    rng = random.Random(SEED)
    for _ in range(CALIBRATION):
        lr = np.load(rng.choice(val_pairs), mmap_mode="r")
        h, w = lr.shape[:2]
        y, x = rng.randint(0, max(h - 128, 0)), rng.randint(0, max(w - 128, 0))
        yield pad_tile(np.array(lr[y:y + 128, x:x + 128]))


def export_tflite(net):
    km, convs, prelus = build_keras()
    port_weights(net.state_dict(), convs, prelus)
    probe = np.random.default_rng(SEED).random((1, 128, 128, 3), dtype=np.float32)
    net.eval()
    with torch.no_grad():
        want = net(torch.from_numpy(probe).permute(0, 3, 1, 2).to(DEVICE)).permute(0, 2, 3, 1).cpu().numpy()
    diff = float(np.abs(km(probe, training=False).numpy() - want).max())
    assert diff < 1e-3, f"bobot Keras tidak cocok: selisih {diff}"
    conv = tf.lite.TFLiteConverter.from_keras_model(km)
    conv.optimizations = [tf.lite.Optimize.DEFAULT]
    conv.representative_dataset = lambda: ([p[None].astype(np.float32) / 255.0] for p in calibration_patches())
    conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    conv.inference_input_type = tf.uint8
    conv.inference_output_type = tf.uint8
    return conv.convert()


def tflite_fn(blob):
    it = tf.lite.Interpreter(model_content=blob, num_threads=THREADS)
    it.allocate_tensors()
    inp, out = it.get_input_details()[0], it.get_output_details()[0]
    in_scale, in_zero = inp["quantization"]
    out_scale, out_zero = out["quantization"]

    def tile(a):
        h, w = a.shape[:2]
        q = np.clip(np.round(pad_tile(a).astype(np.float32) / 255.0 / in_scale + in_zero), 0, 255).astype(np.uint8)
        it.set_tensor(inp["index"], q[None]); it.invoke()
        o = (it.get_tensor(out["index"])[0].astype(np.float32) - out_zero) * out_scale * 255.0
        return np.clip(o, 0, 255).round().astype(np.uint8)[:h * SCALE, :w * SCALE]

    def run(a):
        h, w = a.shape[:2]
        if max(h, w) <= 128:
            return tile(a)
        out_img, step = np.zeros((h * SCALE, w * SCALE, 3), np.uint8), 128 - 2 * OVERLAP
        for y in range(0, h, step):
            for x in range(0, w, step):
                sy, sx = max(0, y - OVERLAP), max(0, x - OVERLAP)
                t = tile(a[sy:min(h, y + step + OVERLAP), sx:min(w, x + step + OVERLAP)])
                ch, cw = min(step, h - y), min(step, w - x)
                out_img[y * SCALE:(y + ch) * SCALE, x * SCALE:(x + cw) * SCALE] = \
                    t[(y - sy) * SCALE:(y - sy + ch) * SCALE, (x - sx) * SCALE:(x - sx + cw) * SCALE]
        return out_img
    return run, inp, out


base_blob, tuned_blob = export_tflite(base), export_tflite(tuned)
tuned_path = SAVE / "real_esrgan_x4v3_lora_w8a8.tflite"
tuned_path.write_bytes(tuned_blob)

base_run, _, _ = tflite_fn(base_blob)
tuned_run, inp, out = tflite_fn(tuned_blob)
assert list(inp["shape"]) == [1, 128, 128, 3] and inp["dtype"] == np.uint8
assert list(out["shape"]) == [1, 512, 512, 3] and out["dtype"] == np.uint8

test_pairs = random.Random(SEED).sample(field_pairs or val_pairs, min(TFLITE_IMAGES * len(PRESETS), len(field_pairs or val_pairs)))
report("TFLite 8-bit, pakai tiling seperti app", test_pairs, [("x4v3 bawaan 8-bit", base_run), ("x4v3 fine-tune 8-bit", tuned_run)])
(SAVE / "hasil.txt").write_text("\n".join(lines), encoding="utf-8")
print("\nTersimpan:", tuned_path, f"({tuned_path.stat().st_size / 1024:.0f} KB)")

## 8. Kalau hasilnya menang

Unduh `real_esrgan_x4v3_lora_w8a8.tflite` dari Drive, lalu masukkan ke `composeApp/src/androidMain/assets/sr/` dan daftarkan di `SuperResolution.Model`. Bentuk input/output sama dengan model bawaan, jadi kode tiling tidak berubah.

Jangan pakai model baru kalau hanya menang di DIV2K tetapi kalah di foto lapangan.